In [ ]:
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt

# Tech magic: Initialize JavaScript for the interactive SHAP plots
shap.initjs()

print("Loading data and the winning model...")
PROCESSED_PATH = "../data/processed"
df = pd.read_csv(f"{PROCESSED_PATH}/attrition_features.csv")

# Recreate our feature set
X = pd.get_dummies(df.drop(columns=['Attrition_Label', 'EmployeeNumber'], errors='ignore'), drop_first=True)

# Load the saved Random Forest model
model_path = "../models/attrition_pipeline.joblib"
model = joblib.load(model_path)

print("Calculating SHAP values (this might take a few seconds)...")
# Using TreeExplainer since our winning model is a Random Forest
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

print("\n--- GLOBAL EXPLANATION ---")
print("What drives attrition across the entire company?")
# Summary plot (Global)
# Note: For Random Forest classifiers, shap_values is a list. Index 1 represents the "Yes" (attrition) class.
shap.summary_plot(shap_values[1], X, plot_type="dot")

print("\n--- LOCAL EXPLANATION ---")
print("Why is Employee #0 flagged?")
# Force plot (Local) for the very first employee in our dataset
# We use matplotlib=True to ensure it renders easily in the notebook
local_plot = shap.force_plot(explainer.expected_value[1], shap_values[1][0,:], X.iloc[0,:], matplotlib=True)
plt.show()